In [5]:
import pandas as pd
import numpy as np
import pickle
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.utils import resample
from scipy.stats import ks_2samp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load clean data (94 features, no debt_settlement_flag_Y)
X_train = pd.read_parquet('acrs_data/X_train.parquet')
X_val   = pd.read_parquet('acrs_data/X_val.parquet')
X_test  = pd.read_parquet('acrs_data/X_test.parquet')

y_train_arr = np.load('acrs_data/y_train.npy')
y_val_arr   = np.load('acrs_data/y_val.npy')
y_test_arr  = np.load('acrs_data/y_test.npy')

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"debt_settlement_flag_Y present: {'debt_settlement_flag_Y' in X_train.columns}")
assert X_train.shape[1] == 82, f"Expected 82 features, got {X_train.shape[1]}"
print("Feature check passed — 82 clean features confirmed")

X_train: (449858, 82) | X_val: (375220, 82) | X_test: (504194, 82)
debt_settlement_flag_Y present: False
Feature check passed — 82 clean features confirmed


In [7]:
# 100k stratified subsample for tuning speed
X_sample, y_sample = resample(X_train, y_train_arr, n_samples=100_000,
                               random_state=42, stratify=y_train_arr)

# Seeded sampler — reproducible tuning 
SEED = 42
sampler = optuna.samplers.TPESampler(seed=SEED)

# XGBoost
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'auc', 'random_state': SEED, 'n_jobs': -1
    }
    return roc_auc_score(y_val_arr,
           XGBClassifier(**params).fit(X_sample, y_sample).predict_proba(X_val)[:,1])

print("Tuning XGBoost (30 trials)...")
xgb_study = optuna.create_study(direction='maximize', sampler=sampler)
xgb_study.optimize(xgb_objective, n_trials=30)
print(f"  Best val AUC: {xgb_study.best_value:.4f} | Params: {xgb_study.best_params}")

# LightGBM
def lgbm_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': SEED, 'n_jobs': -1, 'verbose': -1
    }
    return roc_auc_score(y_val_arr,
           LGBMClassifier(**params).fit(X_sample, y_sample).predict_proba(X_val)[:,1])

print("Tuning LightGBM (30 trials)...")
lgbm_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=30)
print(f"  Best val AUC: {lgbm_study.best_value:.4f} | Params: {lgbm_study.best_params}")

# Random Forest
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth':    trial.suggest_int('max_depth', 5, 15),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': SEED, 'n_jobs': -1
    }
    return roc_auc_score(y_val_arr,
           RandomForestClassifier(**params).fit(X_sample, y_sample).predict_proba(X_val)[:,1])

print("Tuning Random Forest (30 trials)...")
rf_study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
rf_study.optimize(rf_objective, n_trials=30)
print(f"  Best val AUC: {rf_study.best_value:.4f} | Params: {rf_study.best_params}")

Tuning XGBoost (30 trials)...
  Best val AUC: 0.7396 | Params: {'n_estimators': 403, 'max_depth': 5, 'learning_rate': 0.02825376619245168, 'subsample': 0.8615995642588238, 'colsample_bytree': 0.8104131127810936}
Tuning LightGBM (30 trials)...
  Best val AUC: 0.7388 | Params: {'n_estimators': 344, 'max_depth': 7, 'learning_rate': 0.02140648055017374, 'subsample': 0.7410039561734838, 'colsample_bytree': 0.6716918586683731}
Tuning Random Forest (30 trials)...
  Best val AUC: 0.7368 | Params: {'n_estimators': 395, 'max_depth': 11, 'max_features': 'log2'}


In [9]:
print("Training final models on full X_train...")

xgb_final = XGBClassifier(**xgb_study.best_params,
                            eval_metric='auc', random_state=SEED, n_jobs=-1)
xgb_final.fit(X_train, y_train_arr)
print("  XGBoost done")

lgbm_final = LGBMClassifier(**lgbm_study.best_params,
                              random_state=SEED, n_jobs=-1, verbose=-1)
lgbm_final.fit(X_train, y_train_arr)
print("  LightGBM done")

rf_final = RandomForestClassifier(**rf_study.best_params,
                                   random_state=SEED, n_jobs=-1)
rf_final.fit(X_train, y_train_arr)
print("  Random Forest done")

Training final models on full X_train...
  XGBoost done
  LightGBM done
  Random Forest done


In [11]:
def gini(y, p): return 2 * roc_auc_score(y, p) - 1
def ks_stat(y, p): return ks_2samp(p[y==1], p[y==0]).statistic

models = {
    'XGBoost':      xgb_final,
    'LightGBM':     lgbm_final,
    'Random Forest': rf_final,
}

# Collect probabilities
probs_val  = {n: m.predict_proba(X_val)[:,1]  for n, m in models.items()}
probs_test = {n: m.predict_proba(X_test)[:,1] for n, m in models.items()}

# Ensemble
probs_val['Ensemble L1']  = np.mean(list(probs_val.values()),  axis=0)
probs_test['Ensemble L1'] = np.mean(list(probs_test.values()), axis=0)

def metrics_row(y, p):
    return (roc_auc_score(y, p), ks_stat(y, p),
            gini(y, p), f1_score(y, (p >= 0.5).astype(int)))

print("VALIDATION SET (2015)")
print(f"{'Model':<22} {'AUC':>7} {'KS':>7} {'Gini':>7} {'F1':>7}")
print("-" * 58)
for name, p in probs_val.items():
    auc, ks, g, f1 = metrics_row(y_val_arr, p)
    print(f"{name:<22} {auc:>7.4f} {ks:>7.4f} {g:>7.4f} {f1:>7.4f}")

print("\nTEST SET (2016+)")
print(f"{'Model':<22} {'AUC':>7} {'KS':>7} {'Gini':>7} {'F1':>7}")
print("-" * 58)
for name, p in probs_test.items():
    auc, ks, g, f1 = metrics_row(y_test_arr, p)
    print(f"{name:<22} {auc:>7.4f} {ks:>7.4f} {g:>7.4f} {f1:>7.4f}")

print("\nLayer 1 — Hybrid Ensemble COMPLETE")

VALIDATION SET (2015)
Model                      AUC      KS    Gini      F1
----------------------------------------------------------
XGBoost                 0.7432  0.3546  0.4864  0.1070
LightGBM                0.7420  0.3536  0.4841  0.0886
Random Forest           0.7386  0.3474  0.4772  0.0007
Ensemble L1             0.7427  0.3538  0.4853  0.0502

TEST SET (2016+)
Model                      AUC      KS    Gini      F1
----------------------------------------------------------
XGBoost                 0.7071  0.2995  0.4142  0.0727
LightGBM                0.7047  0.2962  0.4094  0.0611
Random Forest           0.7008  0.2881  0.4015  0.0006
Ensemble L1             0.7057  0.2974  0.4115  0.0345

Layer 1 — Hybrid Ensemble COMPLETE


In [13]:
import os
os.makedirs('acrs_data', exist_ok=True)

# Save models
with open('acrs_data/xgb_final.pkl',  'wb') as f: pickle.dump(xgb_final, f)
with open('acrs_data/lgbm_final.pkl', 'wb') as f: pickle.dump(lgbm_final, f)
with open('acrs_data/rf_final.pkl',   'wb') as f: pickle.dump(rf_final, f)

# Save probabilities — val and test for all models + ensemble
for name, key in [('xgb', 'XGBoost'), ('lgbm', 'LightGBM'),
                  ('rf', 'Random Forest'), ('ensemble', 'Ensemble L1')]:
    np.save(f'acrs_data/{name}_prob_val.npy',  probs_val[key])
    np.save(f'acrs_data/{name}_prob_test.npy', probs_test[key])

print("Saved models: xgb_final.pkl, lgbm_final.pkl, rf_final.pkl")
print("Saved probs:  *_prob_val.npy, *_prob_test.npy (xgb, lgbm, rf, ensemble)")
print("Layer 1 complete — ready for Layer 2")

Saved models: xgb_final.pkl, lgbm_final.pkl, rf_final.pkl
Saved probs:  *_prob_val.npy, *_prob_test.npy (xgb, lgbm, rf, ensemble)
Layer 1 complete — ready for Layer 2
